# Beyond one database: scale and NoSQL

Practice notebook for the datafromzero SQL lesson **Beyond one database: scale and NoSQL**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

This notebook runs no NoSQL engine: the documents are small pieces of JSON text that DuckDB can read. Step 5 writes a folder called `orders_by_store` next to the notebook. The `EXPLAIN` cell prints with `print(...)`. Exercise 1 reads the `order_docs` table of step 2, so run the worked example first.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it. Run it again at any time to start over.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Order 1001 as rows

In [ ]:
con.sql("""
SELECT o.order_id, s.store_name, p.product_name, oi.quantity
FROM orders AS o
JOIN stores AS s ON s.store_id = o.store_id
JOIN order_items AS oi ON oi.order_id = o.order_id
JOIN products AS p ON p.product_id = oi.product_id
WHERE o.order_id = 1001
""").show()

### Step 2. The same order as a document, kept under a key

In [ ]:
con.sql("""
CREATE TABLE order_docs (doc_key VARCHAR PRIMARY KEY, doc JSON);
INSERT INTO order_docs VALUES
('order:1001', '{"order_id": 1001, "store": "Campus", "customer": 15, "items": [{"product": "Croissant", "qty": 1}, {"product": "Cold Brew", "qty": 1}, {"product": "Espresso", "qty": 2}]}'),
('order:1002', '{"order_id": 1002, "store": "Old Town", "customer": 46, "items": [{"product": "Croissant", "qty": 1}, {"product": "Turkey Sandwich", "qty": 1}]}'),
('order:1335', '{"order_id": 1335, "store": "Old Town", "items": [{"product": "Americano", "qty": 2}]}');
SELECT COUNT(*) AS documents FROM order_docs
""").show()

### Step 3. Read fields, and a missing one

In [ ]:
con.sql("""
SELECT doc_key, doc->>'$.store' AS store, doc->>'$.customer' AS customer, doc->>'$.items[0].product' AS first_item
FROM order_docs ORDER BY doc_key
""").show()

### Step 4. A lookup by key

In [ ]:
con.sql("""
SELECT doc FROM order_docs WHERE doc_key = 'order:1001'
""").show()

### Step 5. Partition the orders by store

In [ ]:
con.sql("""
COPY (SELECT order_id, customer_id, store_id, ordered_at, status, payment_method FROM orders)
TO 'orders_by_store' (FORMAT parquet, PARTITION_BY (store_id))
""").show()

### Step 6. Read the partitions

In [ ]:
con.sql("""
SELECT store_id, COUNT(*) AS orders
FROM read_parquet('orders_by_store/*/*.parquet', hive_partitioning = true)
GROUP BY store_id ORDER BY store_id
""").show()

In [ ]:
print(con.execute("""
EXPLAIN SELECT COUNT(*)
FROM read_parquet('orders_by_store/*/*.parquet', hive_partitioning = true)
WHERE store_id = 3
""").fetchone()[1])

### Step 7. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders FROM orders
""").show()

## Exercise 1. Read two fields from each document

Finish the query: the store, and the product of the second item (`$.items[1].product`) of each order in `order_docs`.

In [ ]:
con.sql("""
SELECT doc_key, doc->>'$.store' AS store
FROM order_docs
""").show()

## Exercise 2. Which key spreads the orders most evenly?

Before you run anything, write one sentence in the next cell: which of store, month or payment method do you expect to be the most even, and why? Then count orders per store, per month (`strftime(ordered_at, '%Y-%m')`) and per `payment_method`.

In [ ]:
# Expected result, in words:
#

In [ ]:
# Orders per store

In [ ]:
# Orders per month

In [ ]:
# Orders per payment method

## Exercise 3. Spot the flaw

A colleague says: "Orders is our biggest table. Let's shard it by store across three servers." Check the size first, then say what is wrong with the proposal.

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders FROM orders
""").show()

In [ ]:
# What is wrong with the proposal?